# Class 5 · borrow a trained brain (transfer learning)
Instructor: Nazmus Sayadat Rumman. Same rice leaves as Class 4, but this time we start from **MobileNetV2**, a network Google already trained on 1.2 million ImageNet photos. The last cell saves **class5_ship.zip**.

**Before Run All (Add Input):** 1) the RiceLeafBD photos, 2) `rice_split.csv`.
**Settings → Accelerator → GPU** and **Settings → Internet → On** (the ImageNet weights are downloaded). Run All takes about 6 minutes on GPU.

**D5-01 · Setup: same photos, same split, same load_image as Class 4**

In [ ]:
import glob, json, os, shutil, time
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import tensorflow as tf, keras
keras.utils.set_random_seed(42)
CLASSES = ["Blast", "Narrow_Brown_Spot", "Normal_Leaf", "Sheath_Blight", "Tungro"]
hits = glob.glob("/kaggle/input/**/Tungro", recursive=True) + glob.glob("**/Tungro", recursive=True)
assert hits, "RiceLeafBD photos not found. Right panel -> Add Input -> RiceLeafBD dataset, then run this cell again."
ROOT = os.path.dirname(hits[0])
split = pd.read_csv((glob.glob("/kaggle/input/**/rice_split.csv", recursive=True) + ["data/rice_split.csv"])[0])

def load_image(path):                              # identical to Class 4 (the API uses it too)
    img = tf.io.decode_image(tf.io.read_file(path), channels=3, expand_animations=False)
    return tf.image.resize(img, [160, 160])
def make_ds(name):
    df = split[split["split"] == name]
    paths, labels = list(ROOT + "/" + df["cls"] + "/" + df["file"]), df["cls"].map(CLASSES.index).to_numpy()
    ds = tf.data.Dataset.from_tensor_slices((paths, labels)).map(lambda p, y: (load_image(p), y)).cache()
    return (ds.shuffle(3000) if name == "train" else ds).batch(32), paths, labels
(train_ds, _, _), (val_ds, _, y_val), (test_ds, test_paths, y_img) = [make_ds(s) for s in ["train", "val", "test"]]
print("TensorFlow", tf.__version__, "| GPU:", len(tf.config.list_physical_devices("GPU")), "|", split["split"].value_counts().to_dict())

**D5-02 · Ask a famous network: what does ImageNet's MobileNetV2 think our leaf is?**

In [ ]:
famous = keras.applications.MobileNetV2(input_shape=(160, 160, 3), weights="imagenet")   # the full ImageNet model, 1000 classes
leaf_path = sorted(glob.glob(f"{ROOT}/Blast/*.jpg"))[0]
leaf = load_image(leaf_path)
p = famous.predict(keras.applications.mobilenet_v2.preprocess_input(leaf[None]), verbose=0)
top3 = keras.applications.mobilenet_v2.decode_predictions(p, top=3)[0]        # [(id, name, probability), ...]
fig, ax = plt.subplots(1, 2, figsize=(10, 3))
ax[0].imshow(leaf.numpy().astype("uint8")); ax[0].axis("off"); ax[0].set_title("true: Blast")
ax[1].barh([n for _, n, _ in top3][::-1], [q for _, _, q in top3][::-1], color="gray"); ax[1].set_xlim(0, 1)
ax[1].set_title("ImageNet doctor says"); plt.tight_layout(); plt.show()
print("It has never seen a rice disease, but it has learned edges, spots and textures from 1.2 million photos.")

**D5-03 · Borrow its eyes, not its answers: remove the 1000-class top, freeze the rest**

In [ ]:
base = keras.applications.MobileNetV2(input_shape=(160, 160, 3), include_top=False, weights="imagenet")
base.trainable = False                                   # frozen: its 2.2 million weights will not change
print("layers in MobileNetV2:", len(base.layers), "| weights:", f"{base.count_params():,}")
print("output for one photo:", base(tf.zeros((1, 160, 160, 3))).shape, " = 5 x 5 places x 1280 features")

**D5-04 · Our model = borrowed eyes + a new 5-class head**

In [ ]:
model = keras.Sequential([
    keras.Input((160, 160, 3)),
    keras.layers.RandomFlip("horizontal_and_vertical"), keras.layers.RandomRotation(0.1),   # same as Class 4
    keras.layers.Rescaling(1 / 127.5, offset=-1),        # 0..255 -> -1..1, the way MobileNetV2 was trained
    base,
    keras.layers.GlobalAveragePooling2D(),
    keras.layers.Dropout(0.2),
    keras.layers.Dense(5, activation="softmax", name="head")])
trainable = sum(np.prod(w.shape) for w in model.trainable_weights)
print("all weights:", f"{model.count_params():,}", "| weights we train now:", f"{trainable:,}", "(Class 4 CNN trained 99,045)")

**D5-05 · Step 1, feature extraction: train only the head** (base frozen, learning rate 0.001)

In [ ]:
model.compile(optimizer=keras.optimizers.Adam(1e-3), loss="sparse_categorical_crossentropy", metrics=["accuracy"])
stop = keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True)
t0 = time.time()
h1 = model.fit(train_ds, validation_data=val_ds, epochs=20, callbacks=[stop], verbose=2)
b1 = int(np.argmin(h1.history["val_loss"]))
print(f"frozen: {len(h1.history['loss'])} epochs in {time.time() - t0:.0f} s | kept epoch {b1 + 1} | validation accuracy {h1.history['val_accuracy'][b1]:.4f}")

**D5-06 · Step 2, fine-tuning: unfreeze the last blocks, tiny learning rate** (0.00001, 100 times smaller)

In [ ]:
base.trainable = True
start = [l.name for l in base.layers].index("block_13_expand")
for layer in base.layers[:start]:
    layer.trainable = False                              # early layers (edges, colours) stay as they are
for layer in base.layers:
    if isinstance(layer, keras.layers.BatchNormalization):
        layer.trainable = False                          # keep ImageNet's normalisation statistics
trainable = sum(np.prod(w.shape) for w in model.trainable_weights)
print("unfrozen from layer", start, "of", len(base.layers), "| weights we train now:", f"{trainable:,}")
model.compile(optimizer=keras.optimizers.Adam(1e-5), loss="sparse_categorical_crossentropy", metrics=["accuracy"])
h2 = model.fit(train_ds, validation_data=val_ds, epochs=15, callbacks=[keras.callbacks.EarlyStopping(patience=4, restore_best_weights=True)], verbose=2)
b2 = int(np.argmin(h2.history["val_loss"]))
print(f"fine-tuned: kept epoch {b2 + 1} | validation accuracy {h2.history['val_accuracy'][b2]:.4f}")

**D5-07 · The learning curve of both steps** (dashed line = where fine-tuning starts)

In [ ]:
acc = h1.history["accuracy"] + h2.history["accuracy"]; val = h1.history["val_accuracy"] + h2.history["val_accuracy"]
plt.plot(acc, label="train"); plt.plot(val, label="validation"); plt.axvline(len(h1.history["loss"]) - 0.5, ls="--", c="gray")
plt.xlabel("epoch"); plt.ylabel("accuracy"); plt.legend(); plt.show()

**D5-08 · The test set, used once**

In [ ]:
probs = model.predict(test_ds, verbose=0); pred = probs.argmax(1); names = np.array(CLASSES)
print("TEST accuracy:", round((pred == y_img).mean(), 4), "| Class 4 CNN on the same test set: about 0.88")
print(pd.crosstab(names[y_img], names[pred], rownames=["true"], colnames=["MobileNet said"]))

**D5-09 · A doctor who can say "not sure"**: pick the threshold on validation, then check it on test

In [ ]:
pv = model.predict(val_ds, verbose=0)
def answered(p, y, t):                                  # share of photos answered, and accuracy on those
    keep = p.max(1) >= t
    return keep.mean(), (p.argmax(1)[keep] == y[keep]).mean()
ts = np.round(np.arange(0.20, 0.991, 0.01), 2)
t_best = next((t for t in ts if answered(pv, y_val, t)[1] >= 0.98), 0.99)   # lowest threshold with 98% right on answered photos
cover, right = zip(*[answered(pv, y_val, t) for t in ts])
plt.plot(ts, cover, label="share of photos answered"); plt.plot(ts, right, label="accuracy on answered photos")
plt.axvline(t_best, ls="--", c="gray"); plt.xlabel("not-sure threshold"); plt.legend(); plt.title("validation"); plt.show()
for name, p, y in [("validation", pv, y_val), ("test", probs, y_img)]:
    c, r = answered(p, y, t_best)
    print(f"{name:10s} threshold {t_best} | answers {c:.0%} of photos | right on those {r:.1%} | the rest: 'not sure, ask an expert'")

**D5-10 · Report card on 10 test photos** (green = right, red = wrong, orange = "not sure")

In [ ]:
fig, ax = plt.subplots(2, 5, figsize=(16, 7))
for a, i in zip(ax.flat, np.random.default_rng(7).choice(len(y_img), 10, replace=False)):
    sure = probs[i].max() >= t_best
    colour = "orange" if not sure else ("green" if pred[i] == y_img[i] else "red")
    a.imshow(load_image(test_paths[i]).numpy().astype("uint8")); a.axis("off")
    a.set_title(f"true: {names[y_img[i]]}\nsaid: {names[pred[i]] if sure else 'not sure'} ({probs[i].max():.0%})", color=colour, fontsize=10)
plt.tight_layout(); plt.show()

**D5-11 · The price: size and speed**

In [ ]:
os.makedirs("ship/models", exist_ok=True); os.makedirs("ship/data/samples", exist_ok=True)
keras.Model(model.inputs, model.outputs).save("ship/models/rice_tl.keras")    # a copy without the optimizer: 9.6 MB, not 23
one = load_image(test_paths[0])[None]; model.predict(one, verbose=0)
t0 = time.time(); [model.predict(one, verbose=0) for _ in range(20)]
print(f"rice_tl.keras {os.path.getsize('ship/models/rice_tl.keras') / 1e6:.1f} MB | {1000 * (time.time() - t0) / 20:.0f} ms per photo (Class 4 CNN: 1.3 MB)")

**D5-12 · Save and zip: the new model, its meta (with the not-sure threshold), 15 test photos → class5_ship.zip**

In [ ]:
json.dump({"classes": CLASSES, "image_size": [160, 160], "model_file": "rice_tl.keras", "model_name": "MobileNetV2 fine-tuned (Class 5)",
           "not_sure_threshold": float(t_best)}, open("ship/models/rice_meta.json", "w"), indent=1)
for i in [i for c in range(5) for i in np.where(y_img == c)[0][:3]]:      # the same 15 test photos as Class 4
    shutil.copy(test_paths[i], f"ship/data/samples/{CLASSES[y_img[i]]}__{os.path.basename(test_paths[i])}")
    say = names[pred[i]] if probs[i].max() >= t_best else "not sure"
    print(f"{CLASSES[y_img[i]]:18s} MobileNet: {names[pred[i]]:18s} {probs[i].max():.1%}  -> {say}")   # the app must show the same
shutil.make_archive("class5_ship", "zip", "ship"); print("saved class5_ship.zip:", sorted(os.listdir("ship/models")))